<a href="https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/codex/MLR-20-eprime-wandb-integration/notebooks/demo_spotlight_e_prime_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Spotlight E-Prime Constraint Retention In Colab [A2_C0_C2]

Worker-backed MLR-30 / MLR-32 query-preserving Spotlight experiment. This notebook runs one selected condition per kernel (`MLR20_SELECTED_CONDITION_ID`, default `C2`) because vLLM mutates the process-global Worker class when a worker extension loads. Run A2, C0, and C2 in fresh runtimes to produce comparable E-Prime/spaCy/W&B runs. C1 is intentionally excluded because it was invalidated as a patched-notebook/hallucinated condition without a refactored worker behind it.


### Installation

Run this setup cell once in a fresh Colab GPU runtime before continuing. It clones the repo and installs the CUDA-compatible notebook dependencies.


In [1]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
#
# PURPOSE:
# This cell prepares the environment to run vLLM and its custom plugins.
# It handles complex dependency conflicts common in Colab (e.g., pre-installed
# torch versions) and ensures the plugin code is loaded correctly.
#
# KEY ACTIONS:
# 1. Clones the vLLM-Hook repository if not present.
# 2. Installs a compatible version of vLLM and PyTorch for your GPU.
# 3. Cleans up old binary artifacts to prevent version conflicts.
# 4. Loads the plugin source code.
# 5. TRIGGERS A COLAB RESTART: The cell will restart the runtime to ensure
#    the new libraries are fully loaded into the kernel memory.
#
# EXPECTED BEHAVIOR:
# - The cell will run and install packages.
# - It may print "Restarting Colab runtime...".
# - The cell will stop abruptly, and the runtime will restart.
# - The restart is automatic; the code will resume from the top.
# ==============================================================================

from pathlib import Path
import importlib
import importlib.util
import os
import re
import shutil
import site
import subprocess
import sys
import time

# Configuration
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "codex/mlr-30-32-query-preserving-spotlight")
REPO_NAME = "vLLM-Hook"

# Environment Variables (Optional overrides)
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.11,<0.19")
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128")

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()

# --------------------------------------------------------------------------
# Helper Functions
# --------------------------------------------------------------------------

def run(cmd, cwd=None, env=None):
    """
    Executes a shell command, printing output in real-time.
    Raises an error if the command fails.
    """
    cmd = [str(part) for part in cmd]
    print(f"> Running: {' '.join(cmd)}", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail = tail[-120:]

    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )

def run_capture(cmd):
    """Executes a command and returns stdout/stderr without printing."""
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)

def norm(name):
    """Normalize package names for comparison."""
    return name.lower().replace("_", "-")

def package_from_req_line(line: str) -> str:
    """Extract package name from a requirement string (e.g., 'torch>=1.0' -> 'torch')."""
    stripped = line.strip()
    # Split on version specifiers
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())

def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    """Checks if the git repo's origin matches the expected URL."""
    try:
        url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return url == expected_remote

def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    """Searches up the directory tree for a matching git repo."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None

def assert_cuda_runtime():
    """Ensures a GPU is available before proceeding."""
    try:
        import torch
    except Exception:
        torch = None

    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None

    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "No CUDA GPU detected. "
            "In Colab, go to Runtime > Change runtime type and select T4 GPU (or better), "
            "then re-run the entire notebook from the beginning."
        )

# --------------------------------------------------------------------------
# 1. Repository Setup
# --------------------------------------------------------------------------

expected_remote = REPO_URL.removesuffix(".git")
existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)

if IN_COLAB:
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Refreshing existing repo at: {REPO_ROOT}")
        run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch detected. Replacing clone...")
            shutil.rmtree(REPO_ROOT)
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        else:
            print(f"Refreshing existing clone...")
            run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Working directory set to: {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
FILTERED_REQ_FILE = Path("/tmp/vllm_hook_colab_requirements.txt")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_binary_deps_restarted")

print(f"\n--- Environment Summary ---")
print(f"Running in Colab: {IN_COLAB}")
print(f"Repo Root: {REPO_ROOT}")
print(f"Plugin Dir: {PKG_DIR}")

if IN_COLAB:
    assert_cuda_runtime()

# --------------------------------------------------------------------------
# 2. Plugin Directory Validation
# --------------------------------------------------------------------------

if not PKG_DIR.exists():
    raise FileNotFoundError(
        f"Plugin directory not found at {PKG_DIR}. "
        "Please ensure the repository was cloned correctly."
    )

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("git is required but unavailable in this runtime.")

# --------------------------------------------------------------------------
# 3. Dependency Management
# --------------------------------------------------------------------------

if REQ_FILE.exists():
    keep = []
    # These packages are managed by Colab's pre-installed environment.
    # Installing them again can cause conflicts.
    blocked = {"vllm", "torch", "torchvision", "torchaudio", "numpy", "scipy", "protobuf"}

    for line in REQ_FILE.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue

        package = package_from_req_line(stripped)
        if package in blocked:
            print(f"⏭ Skipping managed dependency: {line}")
            continue
        keep.append(line)

    FILTERED_REQ_FILE.write_text("\n".join(keep) + "\n")
    print(f"Installing filtered requirements from {REQ_FILE.name}...")
    run([sys.executable, "-m", "pip", "install", "-r", str(FILTERED_REQ_FILE)])
else:
    print("⚠ Warning: No requirement.txt found; skipping custom dependency installation.")

# Ensure protobuf is at the correct version
print("Ensuring protobuf version compatibility...")
run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])

# --------------------------------------------------------------------------
# 4. vLLM & PyTorch Installation
# --------------------------------------------------------------------------

if COLAB_INSTALL_VLLM:
    print(f"Installing user-specified vLLM: {COLAB_INSTALL_VLLM}")
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    print(f"\nInstalling vLLM and PyTorch for {VLLM_TORCH_BACKEND}...")

    # 1. Uninstall existing conflicting versions
    run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch", "torchvision", "torchaudio"])

    # 2. Manually remove leftover binary artifacts that pip might miss
    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        leftovers = [
            site_path / "vllm", *site_path.glob("vllm-*.dist-info"),
            site_path / "torch", *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision", *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio", *site_path.glob("torchaudio-*.dist-info"),
        ]
        for leftover in leftovers:
            if leftover.exists():
                print(f"  Cleaning up leftover: {leftover.name}")
                if leftover.is_dir():
                    shutil.rmtree(leftover)
                else:
                    leftover.unlink()

    # 3. Force clear GPU memory
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except:
        pass

    # 4. Install using 'uv' (faster than pip) with specific CUDA backend
    print("  Downloading and installing packages with 'uv'...")
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv", "pip", "install",
        "--system",
        VLLM_SPEC,
        "torch", "torchvision", "torchaudio",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

    # 5. Verify Scipy/Numpy
    scipy_check = run_capture([
        sys.executable, "-c", "import numpy, scipy; print('numpy', numpy.__version__); print('scipy', scipy.__version__)"
    ])
    if scipy_check.returncode:
        print("  Detected numpy/scipy issues; upgrading...")
        run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy", "scipy"])

# Verify Installation
print("\nVerifying installation...")
run([
    sys.executable,
    "-c",
    "import torch, vllm; print('✓ torch:', torch.__version__); print('✓ vllm:', getattr(vllm, '__version__', 'unknown'))"
])

# --------------------------------------------------------------------------
# 5. Plugin Loading
# --------------------------------------------------------------------------

# Strategy: Add path to sys.path to allow immediate import,
# then ensure metadata is installed for subprocesses later.
plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

try:
    spec = importlib.util.spec_from_file_location("vllm_hook_plugins", PKG_DIR / "__init__.py")
    if spec:
        importlib.util.module_from_spec(spec)
        print("✓ Plugin module loaded successfully from source path.")
except Exception as e:
    print(f"⚠ Warning: Initial import check failed ({e}). This is expected if compilation is needed later.")

# Ensure the package is registered in the environment (metadata)
# This is necessary for tools that rely on `importlib.metadata`.
print("Registering plugin package in environment...")
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])

print("Installing W&B tracking client...")
run([sys.executable, "-m", "pip", "install", "wandb"])

print(f"Plugin Source: {plugin_src_dir}")
print(f"Python Exec  : {sys.executable}")

# --------------------------------------------------------------------------
# 6. Runtime Restart (Critical for Colab)
# --------------------------------------------------------------------------
# We restart the runtime to ensure the newly installed binary libraries
# are loaded into a fresh Python interpreter. This prevents "dirty state" issues.
if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("\n" + "="*50)
    print("RESTARTING COLAB RUNTIME...")
    print("This ensures the new vLLM/Torch binaries are fully loaded.")
    print("Do not interrupt this process.")
    print("="*50)

    time.sleep(1) # Allow output buffer to flush
    sys.exit(0) # Terminate this process to trigger Colab restart


Refreshing existing clone...
> Running: git -C /content/vLLM-Hook fetch origin codex/mlr-30-32-query-preserving-spotlight
From https://github.com/tburleyinfo/vLLM-Hook
 * branch            codex/mlr-30-32-query-preserving-spotlight -> FETCH_HEAD
   5112c30..2e9dd5c  codex/mlr-30-32-query-preserving-spotlight -> origin/codex/mlr-30-32-query-preserving-spotlight
> Running: git -C /content/vLLM-Hook checkout codex/mlr-30-32-query-preserving-spotlight
Already on 'codex/mlr-30-32-query-preserving-spotlight'
Your branch is behind 'origin/codex/mlr-30-32-query-preserving-spotlight' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
> Running: git -C /content/vLLM-Hook pull --ff-only origin codex/mlr-30-32-query-preserving-spotlight
From https://github.com/tburleyinfo/vLLM-Hook
 * branch            codex/mlr-30-32-query-preserving-spotlight -> FETCH_HEAD
Updating 5112c30..2e9dd5c
Fast-forward
 ..._spotlight_e_prime_query_preserving_colab.ipynb | 216 ++------

### Scoring Dependencies

The E-Prime checker uses spaCy for state-of-being verb detection. Run this cell before scoring.


In [2]:
# Optional scoring dependency setup for Colab.
# Run after the main setup cell. If you already have spaCy and en_core_web_sm, this is a no-op.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("spacy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])

try:
    import spacy
    spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


### Imports & Environment


In [3]:
import gc
import io
import json
import os
import multiprocessing as mp
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from vllm import SamplingParams

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins
from research.experiment_tracking import (
    ExperimentCondition,
    ExperimentResult,
    RunMetrics,
    TurnResult,
    WandbTracker,
    evaluate_mlr20_condition,
    log_mlr20_results_sequentially,
    run_mlr20_batch,
    collect_runtime_provenance,
    format_gib,
    has_required_vram,
    required_vram_bytes,
    turn_result_from_eprime_row,
)

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


Environment configured


### A2/C0/C2 Worker-Backed Implementation Plan


In [4]:
QUERY_PRESERVING_IMPLEMENTATION_MODE = os.environ.get("QUERY_PRESERVING_IMPLEMENTATION_MODE", "A2_C0_C2")
if QUERY_PRESERVING_IMPLEMENTATION_MODE != "A2_C0_C2":
    raise ValueError(f"Unsupported QUERY_PRESERVING_IMPLEMENTATION_MODE={QUERY_PRESERVING_IMPLEMENTATION_MODE!r}")

print("Running worker-backed A2/C0/C2 comparison")
print("A2 -> canonical probe_spotlight")
print("C0 -> canonical probe_spotlight with per-query diagnostic logging")
print("C2 -> query-preserving probe_spotlight_query_preserving")
print("C1 is intentionally excluded: invalidated patched-notebook condition, no refactored worker path")


Running worker-backed A2/C0/C2 comparison
A2 -> canonical probe_spotlight
C0 -> canonical probe_spotlight with per-query diagnostic logging
C2 -> query-preserving probe_spotlight_query_preserving
C1 is intentionally excluded: invalidated patched-notebook condition, no refactored worker path


### Install Canonical Diagnostic Hook For C0


In [5]:
import torch.nn.functional as F
from vllm_hook_plugins.workers.spotlight_worker import SpotlightWorker
from vllm_hook_plugins.utils.spotlight.utils import compute_spotlight_bias as canonical_compute_spotlight_bias

spotlight_query_diagnostics = []
CURRENT_SPOTLIGHT_CONDITION_ID = None

def compute_spotlight_bias_with_c0_diagnostics(logits, span_ranges, target_proportion):
    """Canonical Spotlight plus optional C0 per-query diagnostics."""
    attn_weights = F.softmax(logits, dim=-1)
    if CURRENT_SPOTLIGHT_CONDITION_ID == "C0":
        for batch_idx, ranges in enumerate(span_ranges):
            if not ranges:
                continue
            union_mask = torch.zeros(attn_weights.size(-1), device=attn_weights.device, dtype=attn_weights.dtype)
            for start, end in ranges:
                union_mask[start:end] = 1.0
            union_mask = union_mask.view(1, 1, -1)
            span_mass_by_query = (attn_weights[batch_idx] * union_mask).sum(dim=(0, 2))
            total_mass_by_query = attn_weights[batch_idx].sum(dim=(0, 2)).clamp_min(1e-12)
            psi_current_by_query = span_mass_by_query / total_mass_by_query
            spotlight_query_diagnostics.append({
                "condition": CURRENT_SPOTLIGHT_CONDITION_ID,
                "target_proportion": float(target_proportion),
                "psi_current_by_query": psi_current_by_query.detach().cpu().float().tolist(),
                "psi_current_global": float((attn_weights[batch_idx] * union_mask).sum().detach().cpu() / attn_weights[batch_idx].sum().detach().cpu()),
            })
    return canonical_compute_spotlight_bias(logits, span_ranges, target_proportion)

SpotlightWorker._spotlight_bias_fn = staticmethod(compute_spotlight_bias_with_c0_diagnostics)
print("Canonical Spotlight worker configured with C0-only diagnostics")


Canonical Spotlight worker configured with C0-only diagnostics


### GPU Memory Preflight


In [6]:
def show_nvidia_smi():
    if not IN_COLAB:
        return
    try:
        completed = subprocess.run(
            ["nvidia-smi"],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        print("nvidia-smi unavailable in this runtime.")
        return
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)


def cuda_memory_preflight(gpu_memory_utilization, *, show_smi=True):
    gc.collect()
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. In Colab, select a GPU runtime, restart the runtime, "
            "and run the notebook from the beginning."
        )

    torch.cuda.empty_cache()
    if hasattr(torch.cuda, "ipc_collect"):
        try:
            torch.cuda.ipc_collect()
        except Exception as exc:
            print(f"torch.cuda.ipc_collect() skipped: {exc}")

    free_bytes, total_bytes = torch.cuda.mem_get_info()
    required_bytes = required_vram_bytes(total_bytes, gpu_memory_utilization)
    print(
        "CUDA memory after cleanup: "
        f"free={format_gib(free_bytes)} total={format_gib(total_bytes)} "
        f"required={format_gib(required_bytes)} "
        f"for gpu_memory_utilization={gpu_memory_utilization:.2f}"
    )

    if show_smi:
        show_nvidia_smi()

    if not has_required_vram(
        free_bytes=free_bytes,
        total_bytes=total_bytes,
        gpu_memory_utilization=gpu_memory_utilization,
    ):
        raise RuntimeError(
            "Not enough free CUDA memory for the configured vLLM reservation. "
            f"Free={format_gib(free_bytes)}, required={format_gib(required_bytes)}, "
            f"total={format_gib(total_bytes)}, "
            f"gpu_memory_utilization={gpu_memory_utilization:.2f}. "
            "Restart the Colab runtime and choose Runtime > Run all. "
            "Do not continue from a partially executed runtime."
        )

    return {
        "free_bytes": free_bytes,
        "total_bytes": total_bytes,
        "required_bytes": required_bytes,
    }


### Build `HookLLM` Engines


In [7]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = float(os.environ.get("VLLM_HOOK_GPU_MEMORY_UTILIZATION", "0.30" if IN_COLAB else "0.65"))

gpu_memory_preflight = cuda_memory_preflight(GPU_MEMORY_UTILIZATION)

def build_spotlight_llm(worker_name):
    llm = HookLLM(
        model=model,
        worker_name=worker_name,
        download_dir=cache_dir,
        trust_remote_code=True,
        dtype=torch.float16,
        enable_hook=True,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_model_len=MAX_MODEL_LEN,
        max_num_seqs=1,
        enforce_eager=True,
        enable_prefix_caching=False,
        enable_chunked_prefill=False,
        tensor_parallel_size=1,
    )
    print(f"Model loaded: {model} with worker={worker_name}")
    return llm

print(f"Prepared engine builder for {model}")
print(f"Max model length: {MAX_MODEL_LEN}")
print(f"GPU memory utilization: {GPU_MEMORY_UTILIZATION}")


CUDA memory after cleanup: free=78.83 GiB total=79.25 GiB required=23.78 GiB for gpu_memory_utilization=0.30
Thu Sep 24 11:56:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             60W /  400W |     428MiB /  81920MiB |      0%      Default |
|                              

### E-Prime Long-Conversation Configuration


In [8]:
E_PRIME_CONSTRAINT = """Persistent E-Prime constraint:
Do not use state-of-being verbs: am, is, are, was, were, be, being, been.
Do not use contractions containing state-of-being verbs, including I'm, you're, we're, they're, he's, she's, it's, that's, and there's.
""".strip()

SYSTEM_MESSAGE = (
    "You are a concise assistant continuing a long Talk2AI-style conversation. "
    "Preserve the E-Prime constraint throughout the whole conversation while still answering helpfully."
)

SEED_HISTORY = [
    {"role": "user", "content": "I keep hearing that climate change discussions are exaggerated compared with other problems."},
    {"role": "assistant", "content": "Let's compare long-run measurements with the way headlines describe them."},
    {"role": "user", "content": "The headlines make me distrust the whole topic."},
    {"role": "assistant", "content": "Repeated alarm can make even strong evidence feel performative."},
]

USER_TURNS = [
    "What evidence should I look at if I want to avoid headline-driven conclusions?",
    "How do local weather experiences confuse the broader trend?",
    "What tradeoffs matter for households when policies raise energy costs?",
    "Where does adaptation make sense, and where does it fall short?",
    "How should poorer countries think about growth and emissions limits?",
    "What practical local policy question should I ask a city council candidate?",
    "How can I separate serious policy criticism from misinformation?",
    "What should climate communicators stop doing if they want skeptical people to listen?",
    "What should good-faith skeptics concede before debating policy?",
    "End with a cautious, practical position that still respects the evidence.",
]

ALPHA = 0.2
SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=160)
MAX_USER_TURNS = min(10, len(USER_TURNS))
HISTORY_WINDOW_MESSAGES = 16


### Prompt And Run Helpers


In [9]:
def render_e_prime_prompt(history, user_message):
    recent_history = history[-HISTORY_WINDOW_MESSAGES:]
    omitted_messages = max(0, len(history) - len(recent_history))
    transcript = "\n".join(
        f"{item['role'].upper()}: {item['content']}" for item in recent_history
    )
    if transcript:
        transcript += "\n"

    earlier_context = ""
    if omitted_messages:
        earlier_context = (
            f"Earlier conversation context: {omitted_messages} older messages are omitted "
            "from this prompt to keep the Colab run within memory limits. Continue the same conversation.\n\n"
        )

    return f"""{SYSTEM_MESSAGE}

{E_PRIME_CONSTRAINT}

{earlier_context}Recent conversation:
{transcript}USER: {user_message}
ASSISTANT:""".strip()



STATE_OF_BEING_CORE_WORDS = {"am", "is", "are", "was", "were", "be", "being", "been"}
STATE_OF_BEING_SPACY_MODEL = "en_core_web_sm"
CONTRACTION_PATTERN = re.compile(
    r"\b(?:i'm|you're|we're|they're|he's|she's|it's|that's|there's|what's|who's|where's|when's|why's|how's)\b",
    re.IGNORECASE,
)


def load_state_of_being_nlp(model_name=STATE_OF_BEING_SPACY_MODEL):
    try:
        import spacy
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "spaCy is required for the E-Prime checker. Run the scoring dependency setup cell first."
        ) from exc

    try:
        return spacy.load(model_name, disable=["ner", "parser"])
    except OSError as exc:
        raise OSError(
            f"spaCy model {model_name!r} is not installed. Run the scoring dependency setup cell first."
        ) from exc


state_of_being_nlp = load_state_of_being_nlp()


def is_state_of_being_token(token):
    lemma = token.lemma_.lower()
    text = token.text.lower()
    if lemma == "be" and token.pos_ in {"AUX", "VERB"}:
        return True
    return text in STATE_OF_BEING_CORE_WORDS and token.pos_ in {"AUX", "VERB"}


def check_e_prime_violations(text, nlp=state_of_being_nlp):
    text = text or ""
    doc = nlp(text)
    verb_matches = []
    for token in doc:
        if is_state_of_being_token(token):
            verb_matches.append(
                {
                    "text": token.text,
                    "lemma": token.lemma_,
                    "pos": token.pos_,
                    "tag": token.tag_,
                    "start": token.idx,
                    "end": token.idx + len(token.text),
                }
            )
    contraction_matches = [
        {"text": match.group(0), "start": match.start(), "end": match.end()}
        for match in CONTRACTION_PATTERN.finditer(text)
    ]
    violation_count = len(verb_matches) + len(contraction_matches)
    return {
        "state_of_being_count": len(verb_matches),
        "contraction_count": len(contraction_matches),
        "e_prime_violation_count": violation_count,
        "e_prime_retained": violation_count == 0,
        "e_prime_score": 1.0 if violation_count == 0 else 0.0,
        "state_of_being_matches": verb_matches,
        "contraction_matches": contraction_matches,
    }


def rows_from_experiment_results(experiment_results):
    rows = []
    for experiment_result in experiment_results:
        rows.extend(experiment_result.full_result["rows"])
    return rows


### Run One Worker-Backed Condition Per Runtime


In [10]:
from research.experiment_tracking import mlr20_query_preserving_conditions
from vllm_hook_plugins import generate_with_query_preserving_spotlight

QUERY_PRESERVING_NOTEBOOK = "notebooks/demo_spotlight_e_prime_query_preserving_colab.ipynb"
QUERY_PRESERVING_CONDITIONS = mlr20_query_preserving_conditions(
    temperature=SAMPLING_PARAMS.temperature,
    max_tokens=SAMPLING_PARAMS.max_tokens,
    history_window_messages=HISTORY_WINDOW_MESSAGES,
    notebook=QUERY_PRESERVING_NOTEBOOK,
)
conditions_by_id = {condition.condition_id: condition for condition in QUERY_PRESERVING_CONDITIONS}
if "C1" in conditions_by_id:
    raise AssertionError("C1 must remain excluded; it was invalidated and has no refactored worker")

print("Available worker-backed query-preserving comparison conditions:")
for condition in QUERY_PRESERVING_CONDITIONS:
    print(f"  {condition.condition_id}: implementation={condition.implementation} worker={condition.extra['worker_name']} role={condition.extra['role']}")

# vLLM injects worker-extension attributes into the process-global Worker
# class. Loading probe_spotlight and then probe_spotlight_query_preserving
# in the same kernel raises an attribute-conflict assertion. Run one
# selected condition per fresh runtime/kernel and compare the resulting
# W&B runs/artifacts. Default to C2 for the MLR-30/32 worker-backed test.
SELECTED_CONDITION_ID = os.environ.get("MLR20_SELECTED_CONDITION_ID", "C2")
if SELECTED_CONDITION_ID not in conditions_by_id:
    raise ValueError(f"Unsupported MLR20_SELECTED_CONDITION_ID={SELECTED_CONDITION_ID!r}; expected one of {sorted(conditions_by_id)}")
selected_condition = conditions_by_id[SELECTED_CONDITION_ID]
selected_worker_name = selected_condition.extra["worker_name"]
selected_generate_fn = (
    generate_with_query_preserving_spotlight
    if SELECTED_CONDITION_ID == "C2"
    else generate_with_spotlight
)

def run_condition_with_active_id(llm, condition, generate_fn):
    global CURRENT_SPOTLIGHT_CONDITION_ID
    CURRENT_SPOTLIGHT_CONDITION_ID = condition.condition_id
    try:
        return evaluate_mlr20_condition(
            condition=condition,
            llm=llm,
            sampling_params=SAMPLING_PARAMS,
            seed_history=SEED_HISTORY,
            user_turns=USER_TURNS,
            render_prompt=render_e_prime_prompt,
            score_fn=check_e_prime_violations,
            generate_with_spotlight_fn=generate_fn,
            spotlight_span=E_PRIME_CONSTRAINT,
            provenance=collect_runtime_provenance(REPO_ROOT),
        )
    finally:
        CURRENT_SPOTLIGHT_CONDITION_ID = None

batch_started = time.perf_counter()
print(f"Running {SELECTED_CONDITION_ID} with worker={selected_worker_name}")
llm = build_spotlight_llm(selected_worker_name)
try:
    selected_result = run_condition_with_active_id(llm, selected_condition, selected_generate_fn)
finally:
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

batch_elapsed_s = time.perf_counter() - batch_started
query_preserving_batch_results = [selected_result]
results = pd.DataFrame(rows_from_experiment_results(query_preserving_batch_results))
print(f"{SELECTED_CONDITION_ID} run complete in {batch_elapsed_s:.2f}s")
print(f"Captured {len(spotlight_query_diagnostics)} C0 diagnostic records")
results


Available worker-backed query-preserving comparison conditions:
  A2: implementation=spotlight worker=probe_spotlight role=canonical reference
  C0: implementation=global_diagnostic worker=probe_spotlight role=diagnostic reference
  C2: implementation=query_preserving_spotlight worker=probe_spotlight_query_preserving role=novel MLR-30/32 ablation
Running C2 with worker=probe_spotlight_query_preserving
INFO 09-24 11:56:27 [utils.py:233] non-default args: {'trust_remote_code': True, 'download_dir': '/content/.cache/vllm-hook', 'dtype': torch.float16, 'max_model_len': 8192, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.3, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_extension_cls': 'vllm_hook_plugins.workers.query_preserving_spotlight_worker.QueryPreservingSpotlightWorker', 'model': 'Qwen/Qwen2-1.5B-Instruct'}
WARNING 09-24 11:56:27 [envs.py:1717] Unknown vLLM environment variable detected: VLLM_USE_V1
WARNING 

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


INFO 09-24 11:56:38 [model.py:533] Resolved architecture: Qwen2ForCausalLM
WARNING 09-24 11:56:38 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 09-24 11:56:38 [model.py:1582] Using max model len 8192
WARNING 09-24 11:56:38 [arg_utils.py:2080] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.
INFO 09-24 11:56:38 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 09-24 11:56:38 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-24 11:56:38 [vllm.py:820] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 09-24 11:56:38 [vllm.py:985] Cudagraph is disabled under eager mode
INFO 09-24 11:56:38 [compilation.py:289] Enabled custom fusions: norm_quant, act_quant
INFO 09-24 11:56:40

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-24 11:56:45 [default_loader.py:384] Loading weights took 1.24 seconds
INFO 09-24 11:56:46 [gpu_model_runner.py:4566] Model loading took 2.89 GiB memory and 2.472031 seconds
INFO 09-24 11:56:49 [gpu_worker.py:456] Available KV cache memory: 20.26 GiB
INFO 09-24 11:56:49 [kv_cache_utils.py:1316] GPU KV cache size: 758,848 tokens
INFO 09-24 11:56:49 [kv_cache_utils.py:1321] Maximum concurrency for 8,192 tokens per request: 92.63x
INFO 09-24 11:56:49 [core.py:281] init engine (profile, create kv cache, warmup model) took 2.68 seconds
INFO 09-24 11:56:51 [llm.py:391] Supported tasks: ('generate',)
Model loaded: Qwen/Qwen2-1.5B-Instruct with worker=probe_spotlight_query_preserving


...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

C2 run complete in 39.00s
Captured 0 C0 diagnostic records


,condition,turn,prompt,user_message,model_response,compliant,violation_count,state_of_being_count,contraction_count,score_text_source,prompt_chars,history_messages_in_prompt,latency_s,state_of_being_matches,contraction_matches,reply,e_prime_score,e_prime_retained,e_prime_violation_count
0,C2,1,You are a concise assistant continuing a long ...,What evidence should I look at if I want to av...,Look for studies that have been peer-reviewed ...,False,1,1,0,assistant_response,845,4,0.871864,"[{'text': 'been', 'lemma': 'be', 'pos': 'AUX',...",[],Look for studies that have been peer-reviewed ...,0.0,False,1
1,C2,2,You are a concise assistant continuing a long ...,How do local weather experiences confuse the b...,Local weather experiences can be influenced by...,False,1,1,0,assistant_response,1125,6,1.093337,"[{'text': 'be', 'lemma': 'be', 'pos': 'AUX', '...",[],Local weather experiences can be influenced by...,0.0,False,1
2,C2,3,You are a concise assistant continuing a long ...,What tradeoffs matter for households when poli...,Energy costs can have significant impacts on h...,False,2,1,1,assistant_response,1484,8,1.374330,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 253, 'end': 257}]",Energy costs can have significant impacts on h...,0.0,False,2
3,C2,4,You are a concise assistant continuing a long ...,"Where does adaptation make sense, and where do...",Adaptation can make sense in areas where the c...,False,7,6,1,assistant_response,1925,10,1.833857,"[{'text': 'is', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 276, 'end': 280}]",Adaptation can make sense in areas where the c...,0.0,False,7
4,C2,5,You are a concise assistant continuing a long ...,How should poorer countries think about growth...,Poorer countries may face unique challenges in...,False,3,2,1,assistant_response,2441,12,1.573543,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 281, 'end': 285}]",Poorer countries may face unique challenges in...,0.0,False,3
5,C2,6,You are a concise assistant continuing a long ...,What practical local policy question should I ...,What steps will the candidate take to reduce g...,False,1,1,0,assistant_response,2964,14,0.764222,"[{'text': 'are', 'lemma': 'be', 'pos': 'AUX', ...",[],What steps will the candidate take to reduce g...,0.0,False,1
6,C2,7,You are a concise assistant continuing a long ...,How can I separate serious policy criticism fr...,It's important to distinguish between serious ...,False,12,10,2,assistant_response,3222,16,2.012078,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important to distinguish between serious ...,0.0,False,12
7,C2,8,You are a concise assistant continuing a long ...,What should climate communicators stop doing i...,It's important for climate communicators to be...,False,8,7,1,assistant_response,3758,16,2.114973,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for climate communicators to be...,0.0,False,8
8,C2,9,You are a concise assistant continuing a long ...,What should good-faith skeptics concede before...,It's important for good-faith skeptics to ackn...,False,3,2,1,assistant_response,4212,16,1.860125,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for good-faith skeptics to ackn...,0.0,False,3
9,C2,10,You are a concise assistant continuing a long ...,"End with a cautious, practical position that s...",It's important for policymakers and communicat...,False,7,4,3,assistant_response,4459,16,2.008617,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important for policymakers and communicat...,0.0,False,7


### Validate E-Prime Scoring Scope


In [11]:
if "reply" not in results.columns:
    raise ValueError("Expected generated assistant replies in the results table.")
if not set(["e_prime_score", "e_prime_violation_count", "state_of_being_matches", "contraction_matches"]).issubset(results.columns):
    raise ValueError("Expected assistant-only E-Prime scores from the condition evaluator.")
print("E-Prime compliance was scored only on newly generated assistant replies.")
results[["condition", "turn", "e_prime_score", "e_prime_violation_count", "reply"]].head()


E-Prime compliance was scored only on newly generated assistant replies.


,condition,turn,e_prime_score,e_prime_violation_count,reply
0,C2,1,0.0,1,Look for studies that have been peer-reviewed ...
1,C2,2,0.0,1,Local weather experiences can be influenced by...
2,C2,3,0.0,2,Energy costs can have significant impacts on h...
3,C2,4,0.0,7,Adaptation can make sense in areas where the c...
4,C2,5,0.0,3,Poorer countries may face unique challenges in...


### Compare Constraint Retention


In [12]:
summary = results.groupby("condition").agg(
    mean_e_prime_retention=("e_prime_score", "mean"),
    violation_rate=("e_prime_retained", lambda values: 1 - values.mean()),
    mean_violations=("e_prime_violation_count", "mean"),
    mean_state_of_being_count=("state_of_being_count", "mean"),
    mean_contraction_count=("contraction_count", "mean"),
    mean_latency_s=("latency_s", "mean"),
    max_prompt_chars=("prompt_chars", "max"),
)
summary


,mean_e_prime_retention,violation_rate,mean_violations,mean_state_of_being_count,mean_contraction_count,mean_latency_s,max_prompt_chars
condition,,,,,,,
C2,0.0,1.0,4.5,3.5,1.0,1.550695,4459


### Inspect Failures


In [13]:
failures = results[results["e_prime_score"] < 1.0][
    [
        "condition",
        "turn",
        "user_message",
        "e_prime_violation_count",
        "state_of_being_matches",
        "contraction_matches",
        "reply",
    ]
]
failures


,condition,turn,user_message,e_prime_violation_count,state_of_being_matches,contraction_matches,reply
0,C2,1,What evidence should I look at if I want to av...,1,"[{'text': 'been', 'lemma': 'be', 'pos': 'AUX',...",[],Look for studies that have been peer-reviewed ...
1,C2,2,How do local weather experiences confuse the b...,1,"[{'text': 'be', 'lemma': 'be', 'pos': 'AUX', '...",[],Local weather experiences can be influenced by...
2,C2,3,What tradeoffs matter for households when poli...,2,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 253, 'end': 257}]",Energy costs can have significant impacts on h...
3,C2,4,"Where does adaptation make sense, and where do...",7,"[{'text': 'is', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 276, 'end': 280}]",Adaptation can make sense in areas where the c...
4,C2,5,How should poorer countries think about growth...,3,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 281, 'end': 285}]",Poorer countries may face unique challenges in...
5,C2,6,What practical local policy question should I ...,1,"[{'text': 'are', 'lemma': 'be', 'pos': 'AUX', ...",[],What steps will the candidate take to reduce g...
6,C2,7,How can I separate serious policy criticism fr...,12,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important to distinguish between serious ...
7,C2,8,What should climate communicators stop doing i...,8,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for climate communicators to be...
8,C2,9,What should good-faith skeptics concede before...,3,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for good-faith skeptics to ackn...
9,C2,10,"End with a cautious, practical position that s...",7,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important for policymakers and communicat...


### Save Results, Diagnostics, And Limitations


In [14]:
out_dir = Path("/content/spotlight_e_prime_results") if IN_COLAB else Path("notebooks/results")
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path = out_dir / f"spotlight_e_prime_A2_C0_C2_query_preserving_{stamp}.csv"
json_path = out_dir / f"spotlight_e_prime_A2_C0_C2_query_preserving_{stamp}.json"

results.to_csv(csv_path, index=False)
json_path.write_text(json.dumps({
    "model": model,
    "conditions": [selected_condition.to_config()],
    "available_reference_conditions": [condition.to_config() for condition in QUERY_PRESERVING_CONDITIONS],
    "constraint": E_PRIME_CONSTRAINT,
    "metric": "e_prime_score == 1.0 when no state-of-being verbs or listed contractions appear",
    "max_user_turns": MAX_USER_TURNS,
    "history_window_messages": HISTORY_WINDOW_MESSAGES,
    "max_model_len": MAX_MODEL_LEN,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    "summary": summary.reset_index().to_dict(orient="records"),
    "c0_query_diagnostics": spotlight_query_diagnostics,
    "invalidated_conditions": {"C1": "Excluded; patched-notebook/hallucinated condition without a refactored worker"},
    "limitations": [
        "Notebook smoke validation checks structure and platform-specific imports only unless this notebook is run in Colab.",
        "The contraction checker uses an explicit list and may not catch every informal contraction.",
        "The spaCy POS-based checker can miss malformed output or mis-tag unusual phrasing.",
        "Run one selected condition per fresh runtime because vLLM worker-extension injection is process-global.",
    ],
    "rows": results.to_dict(orient="records"),
    "selected_condition_id": SELECTED_CONDITION_ID,
    "wandb_run_invariant": "one Experimental Conditions row = one ExperimentCondition = one W&B run",
}, indent=2), encoding="utf-8")

print(f"Saved CSV: {csv_path}")
print(f"Saved JSON: {json_path}")
print("Validation note: run this notebook in a Colab GPU runtime for execution results.")


Saved CSV: /content/spotlight_e_prime_results/spotlight_e_prime_A2_C0_C2_query_preserving_20260924-115708.csv
Saved JSON: /content/spotlight_e_prime_results/spotlight_e_prime_A2_C0_C2_query_preserving_20260924-115708.json
Validation note: run this notebook in a Colab GPU runtime for execution results.


### Track A2/C0/C2 As Separate W&B Runs


In [15]:
ENABLE_WANDB = os.environ.get("WANDB_MODE", "").lower() not in {"disabled", "off"}
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "vllm-hook-eprime")
WANDB_API_KEY = ""

if ENABLE_WANDB:
    if WANDB_API_KEY.strip():
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
    elif IN_COLAB:
        try:
            from google.colab import userdata
            wandb_api_key = userdata.get("WANDB_API_KEY")
            if wandb_api_key:
                os.environ["WANDB_API_KEY"] = wandb_api_key
        except Exception:
            pass

for experiment_result in query_preserving_batch_results:
    experiment_result.provenance.update({
        "results_csv": str(csv_path),
        "results_json": str(json_path),
        "batch_elapsed_s": batch_elapsed_s,
        "batch_condition_count": len(query_preserving_batch_results),
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "max_model_len": MAX_MODEL_LEN,
    })
    experiment_result.full_result.update({
        "model": model,
        "constraint": E_PRIME_CONSTRAINT,
        "metric": "e_prime_score == 1.0 when no state-of-being verbs or listed contractions appear",
        "max_user_turns": MAX_USER_TURNS,
        "history_window_messages": HISTORY_WINDOW_MESSAGES,
        "max_model_len": MAX_MODEL_LEN,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "condition_config": experiment_result.condition.to_config(),
        "c0_query_diagnostics": spotlight_query_diagnostics if experiment_result.condition.condition_id == "C0" else [],
        "invalidated_conditions": {"C1": "Excluded; patched-notebook/hallucinated condition without a refactored worker"},
    })

wandb_payloads = log_mlr20_results_sequentially(query_preserving_batch_results, project=WANDB_PROJECT, enabled=ENABLE_WANDB)
print(f"Prepared {len(wandb_payloads)} W&B payload for {SELECTED_CONDITION_ID}")
if ENABLE_WANDB:
    print(f"Logged one W&B run per condition in project={WANDB_PROJECT}")
else:
    print("W&B logging disabled; payloads were built locally for validation.")
wandb_payloads[0]["config"]


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results



KeyboardInterrupt: 